<a href="https://colab.research.google.com/github/rakshamishra54/Amazon_Sales_Analytics/blob/main/Amazon_Sales_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
import pandas as pd , sqlite3
from pathlib import Path

my_drive = Path('/content/drive/MyDrive')
conn = sqlite3.connect('/content/amazon_sales.db')
files = {
    'orders_amazon.csv': 'orders',
    'shipping.csv': 'shipping',
    'payments.csv': 'payments',
    'order_items.csv': 'order_items',
    'customers_amazon.csv': 'customers',
    'inventory.csv': 'inventory',
    'sellers.csv': 'sellers',
    'products.csv': 'products',
    'category.csv': 'category'
}

for filename, table_name in files.items():
    matches = list(my_drive.rglob(filename))
    df = pd.read_csv(matches[0])
    df.columns = df.columns.str.strip()
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f" {table_name}: {len(df)} rows loaded")



 orders: 21629 rows loaded
 shipping: 21141 rows loaded
 payments: 21629 rows loaded
 order_items: 21629 rows loaded
 customers: 898 rows loaded
 inventory: 765 rows loaded
 sellers: 54 rows loaded
 products: 765 rows loaded
 category: 6 rows loaded


In [49]:
##Total number of tables present here
def run(query):
    return pd.read_sql_query(query, conn)

run("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
""")

,name
0,orders
1,shipping
2,payments
3,order_items
4,customers
5,inventory
6,sellers
7,products
8,category


In [50]:
tables = [
    row[0] for row in conn.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name;
    """).fetchall()
]

for table in tables:
    safe_table = '"' + table.replace('"', '""') + '"'
    print(f"\n{'=' * 15} {table.upper()} {'=' * 15}")
    display(run(f"PRAGMA table_info({safe_table});"))


=============== CATEGORY ===============


,cid,name,type,notnull,dflt_value,pk
0,0,category_id,INTEGER,0,None,0
1,1,category_name,TEXT,0,None,0



=============== CUSTOMERS ===============


,cid,name,type,notnull,dflt_value,pk
0,0,Customer ID,INTEGER,0,None,0
1,1,first_name,TEXT,0,None,0
2,2,last_name,TEXT,0,None,0
3,3,state,TEXT,0,None,0



=============== INVENTORY ===============


...

,cid,name,type,notnull,dflt_value,pk
0,0,product_id,INTEGER,0,None,0
1,1,product_name,TEXT,0,None,0
2,2,price,REAL,0,None,0
3,3,cogs,REAL,0,None,0
4,4,category_id,INTEGER,0,None,0



=============== SELLERS ===============


,cid,name,type,notnull,dflt_value,pk
0,0,seller_id,INTEGER,0,None,0
1,1,seller_name,TEXT,0,None,0
2,2,origin,TEXT,0,None,0



=============== SHIPPING ===============


,cid,name,type,notnull,dflt_value,pk
0,0,shipping_id,INTEGER,0,None,0
1,1,order_id,INTEGER,0,None,0
2,2,shipping_date,TEXT,0,None,0
3,3,return_date,TEXT,0,None,0
4,4,shipping providers,TEXT,0,None,0
5,5,delivery_status,TEXT,0,None,0


In [51]:
##Show me 5 columns from the customers table
run("""
SELECT *
FROM customers
LIMIT 5;
""")

,Customer ID,first_name,last_name,state
0,894,Ursula,Lee,Wisconsin
1,895,Samuel,Clark,Wisconsin
2,896,Patrick,Scott,Wisconsin
3,897,Mia,Davis,Wisconsin
4,898,Jake,Davis,Wisconsin


In [52]:
run("""
SELECT *
FROM orders
LIMIT 10;
""")

,order_id,order_date,customer_id,seller_id,order_status
0,5893,2023-11-27,50,5,Inprogress
1,4071,2023-08-29,50,5,Inprogress
2,3721,2023-08-02,50,2,Inprogress
3,4631,2022-01-01,50,5,Inprogress
4,2069,2021-11-21,50,5,Inprogress
5,4951,2021-10-07,50,1,Inprogress
6,3007,2020-06-03,50,1,Inprogress
7,10316,2022-08-03,51,10,Inprogress
8,3547,2022-01-07,51,2,Inprogress
9,13035,2023-07-21,52,4,Inprogress


In [53]:
#Which order statuses appear most frequently
run("""
SELECT order_status, COUNT(*) AS total
FROM orders
GROUP BY order_status
ORDER BY total DESC;
""")

,order_status,total
0,Completed,17802
1,Returned,2840
2,Inprogress,499
3,Cancelled,488


In [54]:
run("""
SELECT DISTINCT seller_id
FROM orders
LIMIT 10;
""")

,seller_id
0,5
1,2
2,1
3,10
4,4
5,14
6,34
7,16
8,47
9,27


In [55]:

run("""
SELECT state , COUNT(*) AS total_customers
FROM customers
GROUP BY state
ORDER BY total_customers DESC;
""")

,state,total_customers
0,California,169
1,Texas,168
2,Ohio,168
3,New York,168
4,Wisconsin,5
5,West Virginia,5
6,Washington,5
7,Virginia,5
8,Vermont,5
9,Utah,5


In [56]:
## Percentage distribution of order statuses
run("""
SELECT order_status,COUNT(*) AS total_orders,
    ROUND(
        COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2
    ) AS percentage
FROM orders
GROUP BY order_status
ORDER BY total_orders DESC;

""")

,order_status,total_orders,percentage
0,Completed,17802,82.31
1,Returned,2840,13.13
2,Inprogress,499,2.31
3,Cancelled,488,2.26


In [67]:
## Are there any duplicate order_ids?
run("""
SELECT COUNT(*) AS total_records,
COUNT(DISTINCT order_id) AS unique_order_ids
FROM orders;
""")

,total_records,unique_order_ids
0,21629,21629


In [58]:
##Which order statuses have fewer than 1,000 records?
run("""
SELECT order_status,COUNT(*) AS total_orders
FROM orders
GROUP BY order_status
HAVING COUNT(*) < 1000
ORDER BY total_orders DESC;

""")

,order_status,total_orders
0,Inprogress,499
1,Cancelled,488


In [59]:
run("""
SELECT MIN(order_date) AS first_order_date,
MAX(order_date) AS  last_order_date
FROM orders;
""")

,first_order_date,last_order_date
0,2020-01-01,2024-07-30


In [60]:
##How many unique customers, sellers, and products exist?
run("""
SELECT
    (SELECT COUNT(DISTINCT customer_id) FROM orders)
        AS unique_order_customers,

    (SELECT COUNT(DISTINCT seller_id) FROM orders)
        AS unique_order_sellers,

    (SELECT COUNT(DISTINCT product_id) FROM "order_items")
        AS products_ordered;
""")

,unique_order_customers,unique_order_sellers,products_ordered
0,686,52,750


In [61]:
##Are any important order fields missing?
run("""
SELECT
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END)
        AS missing_order_id,

    SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END)
        AS missing_order_date,

    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END)
        AS missing_customer_id,

    SUM(CASE WHEN seller_id IS NULL THEN 1 ELSE 0 END)
        AS missing_seller_id,

    SUM(CASE WHEN order_status IS NULL THEN 1 ELSE 0 END)
        AS missing_order_status
FROM orders;
""")

,missing_order_id,missing_order_date,missing_customer_id,missing_seller_id,missing_order_status
0,0,0,0,0,0


In [62]:
##Do any orders reference nonexistent customers?
run("""
SELECT o.order_id , o.customer_id
FROM orders AS o
LEFT JOIN customers AS c
ON o.customer_id = c."Customer ID"
WHERE c."Customer ID" IS NULL ;

""")

,order_id,customer_id


In [66]:
##What is the distribution of payment statuses?
run("""
SELECT payment_status, COUNT(*) AS payment_records
FROM payments
GROUP BY payment_status
ORDER BY payment_records DESC;
""")

,payment_status,payment_records
0,Payment Successed,18301
1,Refunded,2840
2,Payment Failed,488


In [64]:
##How is inventory distributed across warehouses?
run("""
SELECT
    warehouse_id,
    COUNT(*) AS inventory_records,
    SUM(stock) AS total_stock,
    ROUND(AVG(stock), 2) AS average_stock
FROM inventory
GROUP BY warehouse_id
ORDER BY total_stock DESC;
""")

,warehouse_id,inventory_records,total_stock,average_stock
0,1,765,39491,51.62


In [65]:
##Which product categories contain the most products?
run("""
SELECT c.category_name, COUNT(p.product_id) AS product_count
FROM category AS c
LEFT JOIN products AS p
    ON c.category_id = p.category_id
GROUP BY c.category_id, c.category_name
ORDER BY product_count DESC;
""")

,category_name,product_count
0,electronics,308
1,clothing,132
2,home & kitchen,106
3,Pet Supplies,82
4,Sports & Outdoors,69
5,Toys & Games,68
